# Notebook 2 — The Joint Score Field for K=2 (2D Visualization)

**Why K=2?** With only two frames, the joint density $P_t(x_0, x_1)$ is a 2D Gaussian
that we can visualize directly. This lets us see:

1. **The coupling** between $S_0$ and $x_1$ (and vice versa) — absent in the old marginal score
2. **How the score field changes** with diffusion time $t$
3. **The contrast** between the correct joint score and the wrong per-frame marginal score

### Setup

$$P_t(x_0, x_1) = \mathcal{N}\!\left(\begin{pmatrix}x_0\\x_1\end{pmatrix};\;
\mu_t,\; \Sigma_t^{(2)}\right)$$

with $\Sigma_t^{(2)} = e^{-2t}\Sigma_0^{(2)} + \Delta_t I_2$,
and $\Sigma_0^{(2)} = \begin{pmatrix}\sigma_\infty^2 & \alpha\sigma_\infty^2 \\ \alpha\sigma_\infty^2 & \sigma_\infty^2\end{pmatrix}$.

The joint score is:
$$S(x,t) = \begin{pmatrix}S_0\\S_1\end{pmatrix} = -(\Sigma_t^{(2)})^{-1}(x-\mu_t)$$

The **marginal (wrong) score** would be:
$$s_k(x_k,t) = -\frac{x_k - e^{-t}\mu_0}{e^{-2t}\sigma_\infty^2+\Delta_t} \quad\text{(ignores all other frames)}$$

In [ ]:
import sys, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
AUDIT = Path("Research/laplace_ar1_audit/code")
sys.path.insert(0, str(AUDIT))
from ar1_diffusion_utils import (
    gaussian_chain_covariance, gaussian_ou_covariance, gaussian_ou_precision,
    delta_t,
)
plt.rcParams.update({"font.family": "serif", "font.size": 11, "figure.dpi": 120})
print("✓ imports OK")

In [ ]:
# ── Parameters ───────────────────────────────────────────────────────────────
alpha  = 0.8
se2    = 1 - alpha**2       # sigma_eta^2
s0_sq  = 1.0                # sigma_inf^2 = se2/(1-alpha^2) = 1.0
mu0    = 0.0                # zero-mean for simplicity

Sigma0_2 = gaussian_chain_covariance(2, alpha, s0_sq, se2)
print("Σ₀ (K=2):")
print(Sigma0_2)
print(f"\nCorrelation ρ = α·σ∞²/σ∞² = {alpha:.2f}  → entries: {Sigma0_2[0,1]:.4f}")

In [ ]:
# ── Grid for plotting ─────────────────────────────────────────────────────────
x_range = np.linspace(-3.2, 3.2, 200)
X0, X1  = np.meshgrid(x_range, x_range)

def joint_density_2d(X0, X1, Sigma_t, mu_t):
    dx0 = X0 - mu_t[0]
    dx1 = X1 - mu_t[1]
    Q = np.linalg.inv(Sigma_t)
    logdet = np.log(np.linalg.det(Sigma_t))
    quad = Q[0,0]*dx0**2 + 2*Q[0,1]*dx0*dx1 + Q[1,1]*dx1**2
    return np.exp(-0.5*quad) / (2*math.pi * math.sqrt(np.linalg.det(Sigma_t)))

def joint_score_2d(X0, X1, Sigma_t, mu_t):
    Q = np.linalg.inv(Sigma_t)
    dx0 = X0 - mu_t[0]; dx1 = X1 - mu_t[1]
    S0 = -(Q[0,0]*dx0 + Q[0,1]*dx1)
    S1 = -(Q[1,0]*dx0 + Q[1,1]*dx1)
    return S0, S1

def marginal_score_2d(X0, X1, Sigma_t, mu_t):
    "Wrong per-frame score: ignores cross-correlations."
    var0 = Sigma_t[0,0]; var1 = Sigma_t[1,1]
    s0 = -(X0 - mu_t[0]) / var0
    s1 = -(X1 - mu_t[1]) / var1
    return s0, s1

t_vals = [0.05, 0.3, 0.7, 1.5]
fig, axes = plt.subplots(2, len(t_vals), figsize=(14, 8))

for col, t in enumerate(t_vals):
    Sigma_t = gaussian_ou_covariance(Sigma0_2, t)
    mu_t    = math.exp(-t) * np.array([mu0, alpha*mu0])

    P = joint_density_2d(X0, X1, Sigma_t, mu_t)
    S0_joint, S1_joint = joint_score_2d(X0, X1, Sigma_t, mu_t)
    S0_marg,  S1_marg  = marginal_score_2d(X0, X1, Sigma_t, mu_t)

    skip = 14
    ax_top = axes[0, col]
    ax_top.contourf(X0, X1, P, levels=18, cmap="Blues")
    ax_top.quiver(X0[::skip,::skip], X1[::skip,::skip],
                  S0_joint[::skip,::skip], S1_joint[::skip,::skip],
                  color="C1", scale=None, alpha=0.85, width=0.004)
    ax_top.set_title(f"Joint score  $t={t}$", fontsize=11)
    ax_top.set_xlabel("$x_0$"); ax_top.set_ylabel("$x_1$") if col==0 else None

    # Score difference: joint - marginal
    dS0 = S0_joint - S0_marg
    dS1 = S1_joint - S1_marg
    mag = np.sqrt(dS0**2 + dS1**2)
    ax_bot = axes[1, col]
    im = ax_bot.contourf(X0, X1, mag, levels=18, cmap="Reds")
    ax_bot.set_title(f"‖Joint − Marginal‖  $t={t}$", fontsize=11)
    ax_bot.set_xlabel("$x_0$"); ax_bot.set_ylabel("$x_1$") if col==0 else None
    plt.colorbar(im, ax=ax_bot, fraction=0.046)

plt.suptitle(
    rf"K=2 joint density and score (α={alpha}).  "
    r"Top: joint score field.  Bottom: error vs wrong marginal score.",
    y=1.01
)
plt.tight_layout()
plt.savefig(FIG_DIR / "nb2_score_field_k2.png", bbox_inches="tight")
plt.show()
print("Saved → figures/nb2_score_field_k2.png")

## Key observation: the coupling in S₀

In the **marginal score** (old, wrong formulation), $s_0(x_0,t)$ depends **only** on $x_0$.
In the **joint score**, $S_0(x_0,x_1,t)$ depends on **both** $x_0$ and $x_1$:

$$S_0 = -(Q_{00}\,x_0 + Q_{01}\,x_1)$$

The off-diagonal entry $Q_{01} = -(\Sigma_t^{-1})_{01}$ measures how much frame 1's
observation corrects the score for frame 0.

As $t\to 0$: the AR(1) coupling dominates → $Q_{01} \to -\alpha/\sigma_\eta^2 \neq 0$
As $t\to\infty$: OU noise washes out everything → $\Sigma_t \to I$, $Q_{01}\to 0$

In [ ]:
# Show Q_{01}(t) — the off-diagonal coupling strength
t_fine = np.linspace(0.01, 4.0, 300)
Q01_vals = []
for t in t_fine:
    Sigma_t = gaussian_ou_covariance(Sigma0_2, t)
    Q = np.linalg.inv(Sigma_t)
    Q01_vals.append(Q[0,1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(t_fine, Q01_vals, "C0", lw=2)
ax1.axhline(0, color="k", lw=0.8, ls="--")
ax1.set_xlabel("Diffusion time $t$")
ax1.set_ylabel("$Q_{01} = (\Sigma_t^{-1})_{01}$")
ax1.set_title("Cross-frame coupling strength")
ax1.set_xlim(0, 4)

# Compare Q_{00} and Q_{01} at t=0.3
ax2.set_title("Σ_t⁻¹ entries vs t")
Q00_vals = [np.linalg.inv(gaussian_ou_covariance(Sigma0_2, t))[0,0] for t in t_fine]
ax2.plot(t_fine, Q00_vals, "C0", lw=2, label="$Q_{00}$ (self)")
ax2.plot(t_fine, np.abs(Q01_vals), "C1", lw=2, label="$|Q_{01}|$ (cross)")
ax2.set_xlabel("Diffusion time $t$")
ax2.set_ylabel("Entry magnitude")
ax2.legend()
ax2.set_xlim(0, 4)

plt.suptitle(
    rf"K=2 precision matrix entries vs $t$  (α={alpha}). "
    "Cross-coupling disappears at large $t$."
)
plt.tight_layout()
plt.savefig(FIG_DIR / "nb2_coupling_vs_t.png", bbox_inches="tight")
plt.show()

## Score decomposition: what the joint score "does differently"

The bottom row of the main figure shows $\|S_{\rm joint} - S_{\rm marginal}\|$ — the error
of the old per-frame formulation. It is:

- **Large** at small $t$ (strong coupling between frames matters)
- **Largest off-axis** (where $x_0 \neq x_1$, so the correction is asymmetric)
- **Zero** as $t\to\infty$ (both approaches agree when all correlations vanish)

This quantifies exactly how wrong the old score estimates were, and why they would
give a wrong propagator.

## Summary of Notebook 2

- The K=2 joint density is a 2D Gaussian with correlation ρ = α·σ∞²
- The joint score field has **off-diagonal coupling** Q₀₁ ≠ 0 for t > 0
- The marginal (wrong) score misses this coupling entirely
- The error peaks at small t and off-diagonal x values
- As t→∞, both scores coincide (OU erases all correlations)

**Next:** Notebook 3 investigates the Toeplitz structure of Σₜ⁻¹ (H3) and the propagator (H5).